# 01_osteo_manifest_y_subsets

Objetivo:
- localizar imágenes del dataset de Osteoarthritis,
- construir el manifest maestro,
- inferir o crear splits `train/val/test`,
- generar subsets `small`, `baseline` y `large`,
- guardar artefactos reutilizables para las siguientes fases.

## Configuración e imports

In [ ]:
from pathlib import Path
import json
import random
from typing import List, Optional

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ========= RUTAS FIJAS DEL PROYECTO =========
PROJECT_ROOT = Path("/mnt/d/Universidad/analitica/proyecto_analitica2")
DATA_DIR = PROJECT_ROOT / "data"
WORKING_DIR = DATA_DIR / "working"
MANIFESTS_DIR = DATA_DIR / "manifests"
DATA_SOURCE_DIR = DATA_DIR / "source"

# ========= DATASET =========
DATASET_SLUG = "osteo"


DATASET_ROOT = DATA_SOURCE_DIR / DATASET_SLUG / "KLGrade" / "KLGrade"

DATASET_NAME = DATASET_SLUG
LABEL_LEVEL_FROM_IMAGE = 1

KNOWN_SPLIT_NAMES = {
    "train", "training",
    "val", "valid", "validation",
    "test", "testing"
}

IGNORE_FOLDER_NAMES = {
    "images", "image", "img", "imgs",
    "jpg", "jpeg", "png",
    "dataset", "datasets",
    "archive", "data", "source",
    "raw", "processed"
}

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

TEST_SIZE = 0.15
VAL_SIZE = 0.15

SUBSET_FRACTIONS = {
    "small": 0.15,
    "baseline": 0.40,
    "large": 1.00
}

MIN_PER_CLASS = {
    "train": {"small": 20, "baseline": 50, "large": 1},
    "val":   {"small": 8,  "baseline": 15, "large": 1},
    "test":  {"small": 8,  "baseline": 15, "large": 1},
}

META_DIR = WORKING_DIR / DATASET_NAME / "meta"
SUBSETS_DIR = WORKING_DIR / DATASET_NAME / "subsets"

for d in [MANIFESTS_DIR, META_DIR, SUBSETS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("WORKING_DIR:", WORKING_DIR)
print("MANIFESTS_DIR:", MANIFESTS_DIR)
print("DATA_SOURCE_DIR:", DATA_SOURCE_DIR)
print("DATASET_ROOT:", DATASET_ROOT)
print("META_DIR:", META_DIR)
print("SUBSETS_DIR:", SUBSETS_DIR)
print("¿Existe DATASET_ROOT?:", DATASET_ROOT.exists())

PROJECT_ROOT: /mnt/d/Universidad/analitica/proyecto_analitica2
DATA_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data
WORKING_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working
MANIFESTS_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests
DATA_SOURCE_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/source
DATASET_ROOT: /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/osteo/KLGrade/KLGrade
META_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/meta
SUBSETS_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/subsets
¿Existe DATASET_ROOT?: True


## Inspeccionar data/source

In [42]:
print("Contenido de data/source:\n")
for p in sorted(DATA_SOURCE_DIR.iterdir()):
    tipo = "DIR " if p.is_dir() else "FILE"
    print(f"- [{tipo}] {p.name}")

Contenido de data/source:

- [DIR ] isic
- [DIR ] nih
- [DIR ] osteo


In [43]:
if DATASET_ROOT.exists():
    print("Contenido de DATASET_ROOT:\n")
    for p in sorted(DATASET_ROOT.iterdir()):
        tipo = "DIR " if p.is_dir() else "FILE"
        print(f"- [{tipo}] {p.name}")
else:
    print("⚠️ DATASET_ROOT no existe.")
    print('Revisa la línea: DATASET_ROOT = DATA_SOURCE_DIR / "osteo"')

Contenido de DATASET_ROOT:

- [DIR ] 0
- [DIR ] 1
- [DIR ] 2
- [DIR ] 3
- [DIR ] 4


## Helpers

In [44]:
def is_image_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMAGE_EXTS


def normalize_split_name(name: str) -> Optional[str]:
    name = name.lower().strip()
    if name in {"train", "training"}:
        return "train"
    if name in {"val", "valid", "validation"}:
        return "val"
    if name in {"test", "testing"}:
        return "test"
    return None


def sort_labels_nicely(labels: List[str]) -> List[str]:
    def key_fn(x):
        try:
            return (0, int(x))
        except Exception:
            return (1, str(x).lower())
    return sorted(labels, key=key_fn)


def infer_split_from_path(path: Path, dataset_root: Path) -> Optional[str]:
    rel_parts = [p.lower() for p in path.relative_to(dataset_root).parts[:-1]]
    for part in rel_parts:
        split_name = normalize_split_name(part)
        if split_name is not None:
            return split_name
    return None


def infer_label_from_path(path: Path, dataset_root: Path, label_level_from_image: int = 1) -> Optional[str]:
    rel_parts = list(path.relative_to(dataset_root).parts[:-1])  # solo carpetas
    cleaned = []

    for p in rel_parts:
        pl = p.lower().strip()
        if pl in KNOWN_SPLIT_NAMES:
            continue
        if pl in IGNORE_FOLDER_NAMES:
            continue
        cleaned.append(p)

    if len(cleaned) < label_level_from_image:
        return None

    return str(cleaned[-label_level_from_image]).strip()


def get_image_size(path: Path):
    try:
        with Image.open(path) as img:
            return img.size, img.mode
    except Exception:
        return (None, None)


def print_tree(root: Path, max_depth: int = 3, max_entries_per_dir: int = 12, indent: str = ""):
    if not root.exists():
        print(f"[NO EXISTE] {root}")
        return

    if max_depth < 0:
        return

    items = sorted(list(root.iterdir()), key=lambda p: (p.is_file(), p.name.lower()))
    shown = 0
    for item in items:
        if shown >= max_entries_per_dir:
            print(indent + "...")
            break

        print(indent + ("📁 " if item.is_dir() else "📄 ") + item.name)
        shown += 1

        if item.is_dir():
            print_tree(
                item,
                max_depth=max_depth - 1,
                max_entries_per_dir=max_entries_per_dir,
                indent=indent + "    "
            )


def safe_stratified_split(df, test_size, random_state=42):
    label_counts = df["label_text"].value_counts()
    can_stratify = label_counts.min() >= 2

    if can_stratify:
        return train_test_split(
            df,
            test_size=test_size,
            random_state=random_state,
            stratify=df["label_text"]
        )
    else:
        print("⚠️ No se pudo estratificar porque alguna clase tiene menos de 2 muestras. Se hará split aleatorio.")
        return train_test_split(
            df,
            test_size=test_size,
            random_state=random_state,
            stratify=None
        )


def sample_subset_per_split(df_split, fraction, min_per_class, seed=42):
    parts = []

    for label, g in df_split.groupby("label_text"):
        n_total = len(g)
        n_take = max(min_per_class, int(np.ceil(n_total * fraction)))
        n_take = min(n_take, n_total)
        parts.append(g.sample(n=n_take, random_state=seed))

    out = pd.concat(parts, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

In [45]:
image_paths = [p for p in DATASET_ROOT.rglob("*") if is_image_file(p)]
image_paths = sorted(image_paths)

print(f"Total de imágenes encontradas: {len(image_paths):,}")

preview_rows = []
for p in image_paths[:20]:
    rel = p.relative_to(DATASET_ROOT)
    preview_rows.append({
        "rel_path": str(rel),
        "split_inferido": infer_split_from_path(p, DATASET_ROOT),
        "label_lvl1": infer_label_from_path(p, DATASET_ROOT, label_level_from_image=1),
        "label_lvl2": infer_label_from_path(p, DATASET_ROOT, label_level_from_image=2),
        "parent": p.parent.name,
        "grandparent": p.parent.parent.name if p.parent.parent else None
    })

preview_df = pd.DataFrame(preview_rows)
display(preview_df)

Total de imágenes encontradas: 4,766


,rel_path,split_inferido,label_lvl1,label_lvl2,parent,grandparent
0,0/0_110_png.rf.640ac861f605968e142166b62b883aa...,None,0,None,0,KLGrade
1,0/0_111_png.rf.feb50161b2231dc9244620846a45de3...,None,0,None,0,KLGrade
2,0/0_112_png.rf.f0ab25148858fd1db3bdd5934037c7a...,None,0,None,0,KLGrade
3,0/0_114_png.rf.c06380e33be324db4c96b3e0a072b47...,None,0,None,0,KLGrade
4,0/0_115_png.rf.881d3bc5382da08dcd520cfc51304e6...,None,0,None,0,KLGrade
5,0/0_116_png.rf.e3c7074659111cbb1a6dbe43bb4ce70...,None,0,None,0,KLGrade
6,0/0_11_png.rf.db3d03aee2056a5bf54619ae4ce5197e...,None,0,None,0,KLGrade
7,0/0_120_png.rf.7f282c9f8f6a538b82581c8a83fa157...,None,0,None,0,KLGrade
8,0/0_121_png.rf.ecee6abc58f8ecebf6218079e45bc8a...,None,0,None,0,KLGrade
9,0/0_122_png.rf.218033de632b23a1052f205a0bcc92f...,None,0,None,0,KLGrade


## Construir manifest crudo

In [46]:
records = []

for p in image_paths:
    split_name = infer_split_from_path(p, DATASET_ROOT)
    label_text = infer_label_from_path(p, DATASET_ROOT, LABEL_LEVEL_FROM_IMAGE)

    records.append({
        "image_id": p.stem,
        "filename": p.name,
        "file_path": str(p.resolve()),
        "label_text": label_text,
        "split_from_path": split_name,
        "source_dataset": DATASET_NAME
    })

manifest_raw = pd.DataFrame(records)

print("Shape manifest_raw:", manifest_raw.shape)
display(manifest_raw.head())

print("\nConteo de labels inferidas:")
display(
    manifest_raw["label_text"]
    .value_counts(dropna=False)
    .rename_axis("label_text")
    .reset_index(name="count")
)

print("\nConteo de splits inferidos:")
display(
    manifest_raw["split_from_path"]
    .value_counts(dropna=False)
    .rename_axis("split_from_path")
    .reset_index(name="count")
)

Shape manifest_raw: (4766, 6)


,image_id,filename,file_path,label_text,split_from_path,source_dataset
0,0_110_png.rf.640ac861f605968e142166b62b883aa2,0_110_png.rf.640ac861f605968e142166b62b883aa2.jpg,/mnt/d/Universidad/analitica/proyecto_analitic...,0,None,osteo
1,0_111_png.rf.feb50161b2231dc9244620846a45de3e,0_111_png.rf.feb50161b2231dc9244620846a45de3e.jpg,/mnt/d/Universidad/analitica/proyecto_analitic...,0,None,osteo
2,0_112_png.rf.f0ab25148858fd1db3bdd5934037c7ae,0_112_png.rf.f0ab25148858fd1db3bdd5934037c7ae.jpg,/mnt/d/Universidad/analitica/proyecto_analitic...,0,None,osteo
3,0_114_png.rf.c06380e33be324db4c96b3e0a072b47e,0_114_png.rf.c06380e33be324db4c96b3e0a072b47e.jpg,/mnt/d/Universidad/analitica/proyecto_analitic...,0,None,osteo
4,0_115_png.rf.881d3bc5382da08dcd520cfc51304e69,0_115_png.rf.881d3bc5382da08dcd520cfc51304e69.jpg,/mnt/d/Universidad/analitica/proyecto_analitic...,0,None,osteo



Conteo de labels inferidas:


,label_text,count
0,0,1315
1,1,1266
2,2,765
3,3,742
4,4,678



Conteo de splits inferidos:


,split_from_path,count
0,None,4766


## Limpiar manifest y crear target_label

In [47]:
manifest = manifest_raw.copy()

manifest = manifest[manifest["label_text"].notna()].copy()
manifest["label_text"] = manifest["label_text"].astype(str).str.strip()

label_names = sort_labels_nicely(manifest["label_text"].unique().tolist())
label_to_idx = {label: idx for idx, label in enumerate(label_names)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

manifest["target_label"] = manifest["label_text"].map(label_to_idx).astype(int)

print("Labels detectadas:", label_names)
print("label_to_idx:", label_to_idx)
print("Shape manifest limpio:", manifest.shape)
display(manifest.head())

Labels detectadas: ['0', '1', '2', '3', '4']
label_to_idx: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4}
Shape manifest limpio: (4766, 7)


,image_id,filename,file_path,label_text,split_from_path,source_dataset,target_label
0,0_110_png.rf.640ac861f605968e142166b62b883aa2,0_110_png.rf.640ac861f605968e142166b62b883aa2.jpg,/mnt/d/Universidad/analitica/proyecto_analitic...,0,None,osteo,0
1,0_111_png.rf.feb50161b2231dc9244620846a45de3e,0_111_png.rf.feb50161b2231dc9244620846a45de3e.jpg,/mnt/d/Universidad/analitica/proyecto_analitic...,0,None,osteo,0
2,0_112_png.rf.f0ab25148858fd1db3bdd5934037c7ae,0_112_png.rf.f0ab25148858fd1db3bdd5934037c7ae.jpg,/mnt/d/Universidad/analitica/proyecto_analitic...,0,None,osteo,0
3,0_114_png.rf.c06380e33be324db4c96b3e0a072b47e,0_114_png.rf.c06380e33be324db4c96b3e0a072b47e.jpg,/mnt/d/Universidad/analitica/proyecto_analitic...,0,None,osteo,0
4,0_115_png.rf.881d3bc5382da08dcd520cfc51304e69,0_115_png.rf.881d3bc5382da08dcd520cfc51304e69.jpg,/mnt/d/Universidad/analitica/proyecto_analitic...,0,None,osteo,0


## Crear o completar splits

In [48]:
manifest = manifest.copy()

all_have_split = manifest["split_from_path"].notna().all()
unique_splits = set(manifest["split_from_path"].dropna().unique().tolist())

print("¿Todas las filas traen split desde la ruta?", all_have_split)
print("Splits detectados:", unique_splits)

if all_have_split and {"train", "val", "test"}.issubset(unique_splits):
    manifest["split"] = manifest["split_from_path"]
    print("✅ Se usarán los splits detectados desde la ruta.")

elif all_have_split and {"train", "test"}.issubset(unique_splits) and "val" not in unique_splits:
    print("ℹ️ Se detectó train/test pero no val. Se separará val desde train.")

    train_df = manifest[manifest["split_from_path"] == "train"].copy()
    test_df = manifest[manifest["split_from_path"] == "test"].copy()

    val_relative = VAL_SIZE / (1.0 - TEST_SIZE)
    train_df2, val_df = safe_stratified_split(train_df, test_size=val_relative, random_state=SEED)

    train_df2["split"] = "train"
    val_df["split"] = "val"
    test_df["split"] = "test"

    manifest = pd.concat([train_df2, val_df, test_df], axis=0).reset_index(drop=True)

else:
    print("ℹNo se detectaron splits completos. Se crearán desde cero con estratificación si es posible.")

    train_val_df, test_df = safe_stratified_split(manifest, test_size=TEST_SIZE, random_state=SEED)

    val_relative = VAL_SIZE / (1.0 - TEST_SIZE)
    train_df, val_df = safe_stratified_split(train_val_df, test_size=val_relative, random_state=SEED)

    train_df["split"] = "train"
    val_df["split"] = "val"
    test_df["split"] = "test"

    manifest = pd.concat([train_df, val_df, test_df], axis=0).reset_index(drop=True)

manifest = manifest.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("\nDistribución final de split:")
display(manifest["split"].value_counts().rename_axis("split").reset_index(name="count"))

print("\nDistribución final por split y clase:")
display(
    manifest.groupby(["split", "label_text"])
    .size()
    .reset_index(name="count")
    .sort_values(["split", "label_text"])
)

¿Todas las filas traen split desde la ruta? False
Splits detectados: set()
ℹNo se detectaron splits completos. Se crearán desde cero con estratificación si es posible.

Distribución final de split:


,split,count
0,train,3336
1,val,715
2,test,715



Distribución final por split y clase:


,split,label_text,count
0,test,0,197
1,test,1,190
2,test,2,115
3,test,3,111
4,test,4,102
5,train,0,921
6,train,1,886
7,train,2,535
8,train,3,520
9,train,4,474


## Revisar tamaños y modos de algunas imágenes

In [49]:
sample_paths = manifest["file_path"].sample(n=min(30, len(manifest)), random_state=SEED).tolist()

size_rows = []
for fp in sample_paths:
    size, mode = get_image_size(Path(fp))
    size_rows.append({
        "file_path": fp,
        "size": size,
        "mode": mode
    })

sizes_df = pd.DataFrame(size_rows)
display(sizes_df.head(10))

print("Conteo de modos:")
display(
    sizes_df["mode"]
    .value_counts(dropna=False)
    .rename_axis("mode")
    .reset_index(name="count")
)

,file_path,size,mode
0,/mnt/d/Universidad/analitica/proyecto_analitic...,"(300, 162)",RGB
1,/mnt/d/Universidad/analitica/proyecto_analitic...,"(640, 640)",RGB
2,/mnt/d/Universidad/analitica/proyecto_analitic...,"(640, 640)",RGB
3,/mnt/d/Universidad/analitica/proyecto_analitic...,"(640, 640)",RGB
4,/mnt/d/Universidad/analitica/proyecto_analitic...,"(300, 162)",RGB
5,/mnt/d/Universidad/analitica/proyecto_analitic...,"(640, 640)",RGB
6,/mnt/d/Universidad/analitica/proyecto_analitic...,"(640, 640)",RGB
7,/mnt/d/Universidad/analitica/proyecto_analitic...,"(640, 640)",RGB
8,/mnt/d/Universidad/analitica/proyecto_analitic...,"(640, 640)",RGB
9,/mnt/d/Universidad/analitica/proyecto_analitic...,"(300, 162)",RGB


Conteo de modos:


,mode,count
0,RGB,30


## Guardar manifest maestro y label map

In [50]:
master_manifest_path = MANIFESTS_DIR / f"{DATASET_NAME}_master_manifest.csv"
meta_manifest_path = META_DIR / f"{DATASET_NAME}_master_manifest.csv"
label_map_path = META_DIR / f"{DATASET_NAME}_label_map.json"

manifest_to_save = manifest[[
    "image_id",
    "filename",
    "file_path",
    "label_text",
    "target_label",
    "split",
    "source_dataset"
]].copy()

manifest_to_save.to_csv(master_manifest_path, index=False)
manifest_to_save.to_csv(meta_manifest_path, index=False)

with open(label_map_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "label_to_idx": label_to_idx,
            "idx_to_label": {str(k): v for k, v in idx_to_label.items()}
        },
        f,
        ensure_ascii=False,
        indent=2
    )

print("Guardado manifest maestro en:")
print(master_manifest_path)
print(meta_manifest_path)
print(label_map_path)

Guardado manifest maestro en:
/mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests/osteo_master_manifest.csv
/mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/meta/osteo_master_manifest.csv
/mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/meta/osteo_label_map.json


## Crear subsets small, baseline, large

In [51]:
subset_outputs = {}

for subset_name, frac in SUBSET_FRACTIONS.items():
    subset_parts = []

    for split_name in ["train", "val", "test"]:
        df_split = manifest_to_save[manifest_to_save["split"] == split_name].copy()
        min_per_class = MIN_PER_CLASS[split_name][subset_name]

        sampled = sample_subset_per_split(
            df_split=df_split,
            fraction=frac,
            min_per_class=min_per_class,
            seed=SEED
        )
        sampled["subset_name"] = subset_name
        subset_parts.append(sampled)

    subset_df = pd.concat(subset_parts, axis=0).reset_index(drop=True)
    subset_df = subset_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    subset_path = SUBSETS_DIR / f"{DATASET_NAME}_{subset_name}.csv"
    subset_df.to_csv(subset_path, index=False)

    subset_outputs[subset_name] = {
        "df": subset_df,
        "path": subset_path
    }

    print(f"✅ Guardado subset '{subset_name}' en: {subset_path}")
    print(subset_df["split"].value_counts().to_dict())

✅ Guardado subset 'small' en: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/subsets/osteo_small.csv
{'train': 503, 'val': 110, 'test': 110}
✅ Guardado subset 'baseline' en: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/subsets/osteo_baseline.csv
{'train': 1336, 'test': 287, 'val': 287}
✅ Guardado subset 'large' en: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/subsets/osteo_large.csv
{'train': 3336, 'val': 715, 'test': 715}


## Resumen de subsets

In [52]:
for subset_name, obj in subset_outputs.items():
    subset_df = obj["df"]

    print("\n" + "=" * 80)
    print(f"SUBSET: {subset_name.upper()}")
    print(f"Ruta: {obj['path']}")
    print(f"Shape: {subset_df.shape}")

    print("\nConteo por split:")
    display(
        subset_df["split"]
        .value_counts()
        .rename_axis("split")
        .reset_index(name="count")
    )

    print("\nConteo por split y clase:")
    display(
        subset_df.groupby(["split", "label_text"])
        .size()
        .reset_index(name="count")
        .sort_values(["split", "label_text"])
    )


SUBSET: SMALL
Ruta: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/subsets/osteo_small.csv
Shape: (723, 8)

Conteo por split:


,split,count
0,train,503
1,val,110
2,test,110



Conteo por split y clase:


,split,label_text,count
0,test,0,30
1,test,1,29
2,test,2,18
3,test,3,17
4,test,4,16
5,train,0,139
6,train,1,133
7,train,2,81
8,train,3,78
9,train,4,72



SUBSET: BASELINE
Ruta: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/subsets/osteo_baseline.csv
Shape: (1910, 8)

Conteo por split:


...

,split,label_text,count
0,test,0,79
1,test,1,76
2,test,2,46
3,test,3,45
4,test,4,41
5,train,0,369
6,train,1,355
7,train,2,214
8,train,3,208
9,train,4,190



SUBSET: LARGE
Ruta: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/subsets/osteo_large.csv
Shape: (4766, 8)

Conteo por split:


,split,count
0,train,3336
1,val,715
2,test,715



Conteo por split y clase:


,split,label_text,count
0,test,0,197
1,test,1,190
2,test,2,115
3,test,3,111
4,test,4,102
5,train,0,921
6,train,1,886
7,train,2,535
8,train,3,520
9,train,4,474


## subset activo

In [53]:
ACTIVE_SUBSET_NAME = "large"  # cambia entre: small, baseline, large

active_subset_path = subset_outputs[ACTIVE_SUBSET_NAME]["path"]
active_subset_df = subset_outputs[ACTIVE_SUBSET_NAME]["df"].copy()

active_subset_save_path = META_DIR / f"{DATASET_NAME}_active_subset.csv"
active_subset_df.to_csv(active_subset_save_path, index=False)

print("ACTIVE_SUBSET_NAME:", ACTIVE_SUBSET_NAME)
print("active_subset_path:", active_subset_path)
print("active_subset_save_path:", active_subset_save_path)
display(active_subset_df.head())

ACTIVE_SUBSET_NAME: large
active_subset_path: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/subsets/osteo_large.csv
active_subset_save_path: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/meta/osteo_active_subset.csv


,image_id,filename,file_path,label_text,target_label,split,source_dataset,subset_name
0,SevereG4 (233),SevereG4 (233).png,/mnt/d/Universidad/analitica/proyecto_analitic...,4,4,train,osteo,large
1,grade_2_164_png.rf.cf42a3f0b781d99bc4c5c408c8c...,grade_2_164_png.rf.cf42a3f0b781d99bc4c5c408c8c...,/mnt/d/Universidad/analitica/proyecto_analitic...,2,2,train,osteo,large
2,MildG2 (232),MildG2 (232).png,/mnt/d/Universidad/analitica/proyecto_analitic...,2,2,train,osteo,large
3,NormalG0 (64),NormalG0 (64).png,/mnt/d/Universidad/analitica/proyecto_analitic...,0,0,val,osteo,large
4,MildG2 (144),MildG2 (144).png,/mnt/d/Universidad/analitica/proyecto_analitic...,2,2,train,osteo,large


## Chequeos finales

In [54]:
print("Manifest maestro:", master_manifest_path.exists(), master_manifest_path)
print("Label map:", label_map_path.exists(), label_map_path)
print("Active subset:", active_subset_save_path.exists(), active_subset_save_path)

for subset_name, obj in subset_outputs.items():
    print(subset_name, obj["path"].exists(), obj["path"])

missing_mask = ~active_subset_df["file_path"].apply(lambda x: Path(x).exists())
n_missing = int(missing_mask.sum())

print(f"\nArchivos faltantes en active subset: {n_missing}")

if n_missing > 0:
    display(active_subset_df.loc[missing_mask, ["file_path", "label_text", "split"]].head(20))
else:
    print("✅ Todas las rutas del subset activo existen.")

Manifest maestro: True /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests/osteo_master_manifest.csv
Label map: True /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/meta/osteo_label_map.json
Active subset: True /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/meta/osteo_active_subset.csv
small True /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/subsets/osteo_small.csv
baseline True /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/subsets/osteo_baseline.csv
large True /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/osteo/subsets/osteo_large.csv

Archivos faltantes en active subset: 0
✅ Todas las rutas del subset activo existen.
